<a href="https://colab.research.google.com/github/Abish-Devcraft/DSGP-GRP11-SPATIUM/blob/furniture-classifier/notebooks/02_enrichment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 Enrichment
Input: furniture_v1 (MIT crops) from Drive <br>
Output: furniture_v2 (MIT + Open Images + Ceylon Furniture) on Drive

In [2]:
# connect Drive, restore v1 crops and any images downloaded in an earlier session
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, matplotlib.pyplot as plt, json, time, shutil, requests
from pathlib import Path
from PIL import Image

DRIVE = "/content/drive/MyDrive/DSGP"
NEW = Path("/content/new_images")
!unzip -q -o {DRIVE}/furniture_v1.zip -d /content
if Path(f"{DRIVE}/new_images.zip").exists():
    !unzip -q -o {DRIVE}/new_images.zip -d /content

v1 = pd.read_csv(f"{DRIVE}/furniture_v1_manifest.csv")
print(len(v1), "MIT crops loaded")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1396 MIT crops loaded


In [3]:
# how many images each class still needs to reach the targets
CLASSES = ["sofa", "bed", "chair", "wardrobe", "armchair", "dresser", "table", "side_table",
           "coffee_table", "console_table", "tv_stand", "dressing_table", "nightstand", "desk",
           "bookshelf", "storage_cabinet", "stool_ottoman"]
MIN_N, MAX_N, MIN_ROOM, PER_PRODUCT = 150, 300, 30, 3

cnt = v1["cls"].value_counts()
gap = pd.DataFrame({"mit": [cnt.get(c, 0) for c in CLASSES]}, index=CLASSES)
gap["need_total"] = (MAX_N - gap["mit"]).clip(lower=0)
gap["need_room"] = (MIN_ROOM - gap["mit"]).clip(lower=0)
gap.sort_values("mit")

,mit,need_total,need_room
tv_stand,0,300,30
dressing_table,0,300,30
console_table,0,300,30
desk,11,289,19
coffee_table,18,282,12
storage_cabinet,31,269,0
stool_ottoman,35,265,0
nightstand,48,252,0
side_table,50,250,0
dresser,51,249,0


In [4]:
# download real room photos for weak classes and crop each labelled object
!pip install -q fiftyone
import fiftyone.zoo as foz

OI_MAP = {"Nightstand": "nightstand", "Desk": "desk", "Bookcase": "bookshelf",
          "Coffee table": "coffee_table", "Stool": "stool_ottoman", "Cupboard": "storage_cabinet",
          "Chest of drawers": "dresser", "Wardrobe": "wardrobe"}
MIN_SIDE = 48

rows = []
for oi_name, cls in OI_MAP.items():
    target = int(min(gap.loc[cls, "need_total"], 150))
    if target == 0:
        continue
    ds = foz.load_zoo_dataset("open-images-v7", split="train", label_types=["detections"],
                              classes=[oi_name], max_samples=target * 2, only_matching=True,
                              shuffle=True, seed=42, dataset_name=f"oi_{cls}",
                              drop_existing_dataset=True)
    out = NEW / cls
    out.mkdir(parents=True, exist_ok=True)
    n = 0
    for s in ds:
        if n >= target:
            break
        im = Image.open(s.filepath).convert("RGB")
        W, H = im.size
        for k, d in enumerate(s.ground_truth.detections):
            if d.label != oi_name or getattr(d, "IsGroupOf", False) or getattr(d, "IsDepiction", False):
                continue
            x, y, w, h = d.bounding_box
            box = (int(x * W), int(y * H), int((x + w) * W), int((y + h) * H))
            if min(box[2] - box[0], box[3] - box[1]) < MIN_SIDE:
                continue
            p = out / f"oi_{s.id}_{k}.jpg"
            im.crop(box).save(p)
            rows.append({"path": str(p), "cls": cls, "source": "open_images_v7", "group_id": f"oi_{s.id}",
                         "room_crop": True, "width": box[2] - box[0], "height": box[3] - box[1]})
            n += 1
            if n >= target:
                break
    print(f"{cls:16} {n} crops")

oi_df = pd.DataFrame(rows)
oi_df.to_csv(f"{DRIVE}/oi_manifest.csv", index=False)
!cd /content && zip -q -r {DRIVE}/new_images.zip new_images

INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


Necessary images already downloaded


INFO:fiftyone.utils.openimages:Necessary images already downloaded


Existing download of split 'train' is sufficient


INFO:fiftyone.zoo.datasets:Existing download of split 'train' is sufficient


Deleting existing dataset 'oi_nightstand'


... (103 outputs left out)

INFO:eta.core.utils: 100% |█████████████████| 208/208 [388.3ms elapsed, 0s remaining, 535.7 samples/s]     


Dataset 'oi_wardrobe' created


INFO:fiftyone.zoo.datasets:Dataset 'oi_wardrobe' created


wardrobe         150 crops
